In [1]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Feature libraries
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops, label as sk_label
import mahotas

# Paths
PROC = Path('../data/processed')
MASK = Path('../data/masks')
FEAT = Path('../features')
FEAT.mkdir(exist_ok=True)

CLASSES = ['glioma', 'meningioma', 'pituitary', 'notumor']

# Quick sanity check
print("scikit-image features:", graycoprops.__module__)
print("mahotas:", mahotas.__version__)
print("Processed dir exists:", PROC.exists())
print("Mask dir exists:", MASK.exists())

scikit-image features: skimage.feature.texture
mahotas: 1.4.18
Processed dir exists: True
Mask dir exists: True


In [2]:
def extract_first_order(img, mask):
    """
    First-order statistical features computed on the masked region only.
    Returns ~10 features describing pixel intensity distribution.
    """
    pixels = img[mask > 0].astype(np.float64)
    
    if len(pixels) == 0:
        return {f'fo_{k}': 0.0 for k in 
                ['mean', 'std', 'min', 'max', 'median', 'p10', 'p90', 
                 'skew', 'kurt', 'energy', 'entropy', 'range', 'iqr']}
    
    p10, p25, p50, p75, p90 = np.percentile(pixels, [10, 25, 50, 75, 90])
    
    # Skewness and kurtosis (manual, no scipy needed)
    mean = pixels.mean()
    std = pixels.std() + 1e-9
    skew = ((pixels - mean) ** 3).mean() / (std ** 3)
    kurt = ((pixels - mean) ** 4).mean() / (std ** 4) - 3
    
    # Entropy: Shannon entropy of the histogram
    hist, _ = np.histogram(pixels, bins=64, range=(0, 256), density=True)
    hist = hist[hist > 0]
    entropy = -np.sum(hist * np.log2(hist))
    
    return {
        'fo_mean':    float(mean),
        'fo_std':     float(pixels.std()),
        'fo_min':     float(pixels.min()),
        'fo_max':     float(pixels.max()),
        'fo_median':  float(p50),
        'fo_p10':     float(p10),
        'fo_p90':     float(p90),
        'fo_skew':    float(skew),
        'fo_kurt':    float(kurt),
        'fo_energy':  float((pixels ** 2).sum() / 1e6),  # scaled to keep numbers small
        'fo_entropy': float(entropy),
        'fo_range':   float(pixels.max() - pixels.min()),
        'fo_iqr':     float(p75 - p25),
    }


# Test
test_img_path = next((PROC / 'Training' / 'glioma').glob('*.png'))
test_mask_path = MASK / 'Training' / 'glioma' / f'{test_img_path.stem}_mask.png'
test_img = cv2.imread(str(test_img_path), cv2.IMREAD_GRAYSCALE)
test_mask = cv2.imread(str(test_mask_path), cv2.IMREAD_GRAYSCALE)

fo = extract_first_order(test_img, test_mask)
print(f"Got {len(fo)} first-order features:")
for k, v in fo.items():
    print(f"  {k:<15} = {v:.4f}")

Got 13 first-order features:
  fo_mean         = 86.5860
  fo_std          = 49.9146
  fo_min          = 1.0000
  fo_max          = 250.0000
  fo_median       = 80.0000
  fo_p10          = 20.0000
  fo_p90          = 157.0000
  fo_skew         = 0.5022
  fo_kurt         = 0.0132
  fo_energy       = 424.9855
  fo_entropy      = 1.8772
  fo_range        = 249.0000
  fo_iqr          = 60.0000


In [3]:
def extract_glcm(img, mask, levels=32):
    """
    GLCM features: contrast, homogeneity, energy, correlation, dissimilarity.
    Computed at multiple distances and angles, then averaged.
    """
    # Quantize image to fewer gray levels (standard practice for GLCM)
    img_q = (img.astype(np.float32) * (levels - 1) / 255).astype(np.uint8)
    
    # Apply mask: set background pixels to 0
    img_q = np.where(mask > 0, img_q, 0)
    
    # Compute GLCM at distances 1 and 3, four angles each
    glcm = graycomatrix(img_q,
                        distances=[1, 3],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=levels,
                        symmetric=True,
                        normed=True)
    
    feats = {}
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation', 'dissimilarity', 'ASM']:
        vals = graycoprops(glcm, prop)  # shape (distances, angles)
        feats[f'glcm_{prop}_mean'] = float(vals.mean())
        feats[f'glcm_{prop}_std']  = float(vals.std())
    
    return feats


# Test
glcm_feats = extract_glcm(test_img, test_mask)
print(f"Got {len(glcm_feats)} GLCM features:")
for k, v in glcm_feats.items():
    print(f"  {k:<27} = {v:.4f}")

Got 12 GLCM features:
  glcm_contrast_mean          = 15.5159
  glcm_contrast_std           = 7.9130
  glcm_homogeneity_mean       = 0.5281
  glcm_homogeneity_std        = 0.0636
  glcm_energy_mean            = 0.2013
  glcm_energy_std             = 0.0099
  glcm_correlation_mean       = 0.8232
  glcm_correlation_std        = 0.0904
  glcm_dissimilarity_mean     = 2.1774
  glcm_dissimilarity_std      = 0.6680
  glcm_ASM_mean               = 0.0406
  glcm_ASM_std                = 0.0040


In [4]:
def extract_haralick(img, mask):
    """
    Haralick texture features (13 of them).
    Computed only on the masked region by zeroing out background.
    """
    img_masked = np.where(mask > 0, img, 0).astype(np.uint8)
    
    try:
        # ignore_zeros=True skips background pixels
        h = mahotas.features.haralick(img_masked, ignore_zeros=True, return_mean=True)
    except Exception:
        # Some images may have issues; return zeros as fallback
        h = np.zeros(13)
    
    haralick_names = [
        'angular_2nd_moment', 'contrast', 'correlation', 'sum_of_squares',
        'inverse_diff_moment', 'sum_avg', 'sum_variance', 'sum_entropy',
        'entropy', 'difference_variance', 'difference_entropy',
        'info_correlation_1', 'info_correlation_2'
    ]
    
    return {f'har_{n}': float(v) for n, v in zip(haralick_names, h)}


# Test
har_feats = extract_haralick(test_img, test_mask)
print(f"Got {len(har_feats)} Haralick features:")
for k, v in har_feats.items():
    print(f"  {k:<35} = {v:.4f}")

Got 13 Haralick features:
  har_angular_2nd_moment              = 0.0013
  har_contrast                        = 554.0916
  har_correlation                     = 0.8879
  har_sum_of_squares                  = 2474.3352
  har_inverse_diff_moment             = 0.1551
  har_sum_avg                         = 173.8656
  har_sum_variance                    = 9343.2491
  har_sum_entropy                     = 8.3643
  har_entropy                         = 12.9915
  har_difference_variance             = 0.0001
  har_difference_entropy              = 5.2687
  har_info_correlation_1              = -0.2512
  har_info_correlation_2              = 0.9871


In [5]:
def extract_shape(mask):
    """
    Shape features from the binary mask.
    """
    mask_bin = (mask > 0).astype(np.uint8)
    
    # Pick the largest connected region
    labels = sk_label(mask_bin)
    if labels.max() == 0:
        return {f'sh_{k}': 0.0 for k in 
                ['area', 'perimeter', 'eccentricity', 'solidity',
                 'extent', 'major_axis', 'minor_axis', 'circularity', 'aspect_ratio']}
    
    regions = regionprops(labels)
    region = max(regions, key=lambda r: r.area)
    
    area = region.area
    perim = region.perimeter + 1e-9
    circularity = 4 * np.pi * area / (perim ** 2)
    aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
    
    return {
        'sh_area':         float(area),
        'sh_perimeter':    float(region.perimeter),
        'sh_eccentricity': float(region.eccentricity),
        'sh_solidity':     float(region.solidity),
        'sh_extent':       float(region.extent),
        'sh_major_axis':   float(region.major_axis_length),
        'sh_minor_axis':   float(region.minor_axis_length),
        'sh_circularity':  float(circularity),
        'sh_aspect_ratio': float(aspect),
    }


# Test
shape_feats = extract_shape(test_mask)
print(f"Got {len(shape_feats)} shape features:")
for k, v in shape_feats.items():
    print(f"  {k:<20} = {v:.4f}")

C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.minor_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_minor_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)


Got 9 shape features:
  sh_area              = 42547.0000
  sh_perimeter         = 829.9066
  sh_eccentricity      = 0.2095
  sh_solidity          = 0.9594
  sh_extent            = 0.8480
  sh_major_axis        = 236.3783
  sh_minor_axis        = 231.1327
  sh_circularity       = 0.7763
  sh_aspect_ratio      = 1.0227


C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:28: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  'sh_major_axis':   float(region.major_axis_length),
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:29: FutureWarning: `RegionProperties.minor_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_minor_length` instead. 
  'sh_minor_axis':   float(region.minor_axis_length),


In [6]:
def extract_all_features(img_path, mask_path):
    """
    Compute all radiomics features for one image.
    Returns a dict of ~50 features, or None if extraction fails.
    """
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        return None
    if (mask > 0).sum() < 100:  # skip nearly-empty masks
        return None
    
    feats = {}
    feats.update(extract_first_order(img, mask))
    feats.update(extract_glcm(img, mask))
    feats.update(extract_haralick(img, mask))
    feats.update(extract_shape(mask))
    
    return feats


# Test on one image
all_feats = extract_all_features(test_img_path, test_mask_path)
print(f"Total features per image: {len(all_feats)}")
print(f"\nFirst 10 features:")
for i, (k, v) in enumerate(all_feats.items()):
    if i >= 10: break
    print(f"  {k:<27} = {v:.4f}")

Total features per image: 47

First 10 features:
  fo_mean                     = 86.5860
  fo_std                      = 49.9146
  fo_min                      = 1.0000
  fo_max                      = 250.0000
  fo_median                   = 80.0000
  fo_p10                      = 20.0000
  fo_p90                      = 157.0000
  fo_skew                     = 0.5022
  fo_kurt                     = 0.0132
  fo_energy                   = 424.9855


C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.minor_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_minor_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:28: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  'sh_major_axis':   float(region.major_axis_length),
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:29: FutureWarnin

In [7]:
print("Extracting radiomics features for all images...")
print("This will take 15-40 minutes. The progress bars will tell you.")
print()

all_rows = []
fail_count = 0

for split in ['Training', 'Testing']:
    for c in CLASSES:
        in_dir = PROC / split / c
        files = list(in_dir.glob('*.png'))
        
        for f in tqdm(files, desc=f'{split}/{c}'):
            mask_path = MASK / split / c / f'{f.stem}_mask.png'
            
            feats = extract_all_features(f, mask_path)
            if feats is None:
                fail_count += 1
                continue
            
            feats['image_id'] = f.stem
            feats['split']    = split
            feats['class']    = c
            all_rows.append(feats)

print(f"\nDone!")
print(f"Successfully extracted: {len(all_rows)} images")
print(f"Failed (skipped):       {fail_count}")

Extracting radiomics features for all images...
This will take 15-40 minutes. The progress bars will tell you.



Training/glioma:   0%|          | 0/1400 [00:00<?, ?it/s]

C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:20: FutureWarning: `RegionProperties.minor_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_minor_length` instead. 
  aspect = region.major_axis_length / (region.minor_axis_length + 1e-9)
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:28: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  'sh_major_axis':   float(region.major_axis_length),
C:\Users\HP\AppData\Local\Temp\ipykernel_36016\776770108.py:29: FutureWarnin

Training/meningioma:   0%|          | 0/1400 [00:00<?, ?it/s]

Training/pituitary:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Testing/glioma:   0%|          | 0/400 [00:00<?, ?it/s]

Testing/meningioma:   0%|          | 0/400 [00:00<?, ?it/s]

Testing/pituitary:   0%|          | 0/400 [00:00<?, ?it/s]

Testing/notumor:   0%|          | 0/400 [00:00<?, ?it/s]


Done!
Successfully extracted: 7200 images
Failed (skipped):       0


In [8]:
df = pd.DataFrame(all_rows)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.shape[1]} (expected ~50 features + 3 metadata)")

# Move metadata columns to the front for readability
meta_cols = ['image_id', 'split', 'class']
feat_cols = [c for c in df.columns if c not in meta_cols]
df = df[meta_cols + feat_cols]

# Save
out_path = FEAT / 'radiomics.csv'
df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")

DataFrame shape: (7200, 50)
Columns: 50 (expected ~50 features + 3 metadata)

Saved to: C:\Users\HP\Desktop\MU\bmi\brain_tumor_fusion\features\radiomics.csv
File size: 5737.1 KB


In [9]:
# Check for NaN or infinite values
nan_cols = df[feat_cols].isna().sum()
inf_cols = (~np.isfinite(df[feat_cols])).sum()
print("Columns with NaN values:")
print(nan_cols[nan_cols > 0] if nan_cols.sum() > 0 else "  none")
print("\nColumns with Inf values:")
print(inf_cols[inf_cols > 0] if inf_cols.sum() > 0 else "  none")

# Per-class summary of one feature to see if classes differ
print("\nMean values of GLCM contrast across classes (should differ):")
print(df.groupby('class')['glcm_contrast_mean'].agg(['mean', 'std']))

print("\nMean values of first-order entropy across classes:")
print(df.groupby('class')['fo_entropy'].agg(['mean', 'std']))

Columns with NaN values:
  none

Columns with Inf values:
  none

Mean values of GLCM contrast across classes (should differ):
                 mean        std
class                           
glioma      14.085897   3.120310
meningioma  14.983461   3.690277
notumor     27.191831  14.330678
pituitary   17.566792   2.351978

Mean values of first-order entropy across classes:
                mean       std
class                         
glioma      1.869936  0.057559
meningioma  1.917257  0.032229
notumor     1.934983  0.044937
pituitary   1.898719  0.027485


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Split
train = df[df['split'] == 'Training']
test  = df[df['split'] == 'Testing']

X_train = train[feat_cols].values
X_test  = test[feat_cols].values

le = LabelEncoder()
y_train = le.fit_transform(train['class'])
y_test  = le.transform(test['class'])

# Scale
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

# Train baseline RF
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
pred = rf.predict(X_test_sc)

print(f"Radiomics-only baseline accuracy: {accuracy_score(y_test, pred):.4f}")
print()
print(classification_report(y_test, pred, target_names=le.classes_))

Radiomics-only baseline accuracy: 0.8825

              precision    recall  f1-score   support

      glioma       0.94      0.67      0.78       400
  meningioma       0.81      0.93      0.87       400
     notumor       0.89      0.99      0.94       400
   pituitary       0.92      0.94      0.93       400

    accuracy                           0.88      1600
   macro avg       0.89      0.88      0.88      1600
weighted avg       0.89      0.88      0.88      1600

